In [103]:
import requests
import csv
import pandas as pd
import math
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import random
from scipy.stats import linregress
import statsmodels.api as sm
from statsmodels.stats.anova import anova_lm
import pyarrow
import bs4 as bs
import time
import random
from numpy import log as ln

In [ ]:
# Example: 4 risky names + cash (SPY excluded; cash weight 0, cash price 1)
w0 = np.array([[-0.23], [-0.07], [0.8], [0.05], [0.0]])
mr0 = np.array([[0.5], [0.5], [0.8], [1.0], [1.0]])
I = 9000
p0 = np.array([[45], [20], [94], [1], [1]])
p1 = np.array([[51], [29], [113], [1], [1]])
p2 = np.array([[36], [40], [110], [1], [1]])
long_fee = 0.06 / 365
short_fee = 0.06 / 365

In [104]:
def get_prices(ticker, size='full', date_start=None, date_end=None):
    """ Retrieve time series data for the price of a security in data frame format
        Sorts so that first date is first index, filters on date if given, renames columns and converts datatypes

    Args:
        ticker (string): The ticker for the security to pull data from
        size (string): 'full' since 1999, 'compact' is last 100 trading days
        date_filter (date string): Filters  data to  keep data only the filter date onward

    Returns:
        df: A dataframe with open, close, high, low, adj close price, volume, split coef, date
    """
    key='&apikey=HI5ADT0RWWANGUID' # API Key
    api_ticker=f'&symbol={ticker}' # Ticker
    endpoint='function=TIME_SERIES_MONTHLY_ADJUSTED' # Called 'function', the dataset we want
    size=f'&outputsize={size}'
    web='https://www.alphavantage.co/query?'
    url =web+endpoint+api_ticker+size+key

    r = requests.get(url)
    # print(r.status_code) # 200 good, 400 bad
    data = r.json()

    # print(data.keys()) #printing the keys
    meta = data['Meta Data']
    time_series_data = data['Monthly Adjusted Time Series']

    ts_df = pd.DataFrame.from_dict(time_series_data, orient='index').reset_index().rename(columns={'index': 'Date'})
    clean_cols_dict = {'1. open': 'Open', '2. high': 'High', '3. low': 'Low', '4. close': 'Close', # Dictionary to convert the names of the columns
                '5. adjusted close': 'Adj Close', '6. volume': 'Volume', '7. dividend amount': 'Dividend', '8. split coefficient': 'Split Coef'}
    ts_df = ts_df.rename(columns=clean_cols_dict)

    ts_df['Date'] = pd.to_datetime(ts_df['Date'])
    if date_start is not None:
        date_start = pd.to_datetime(date_start)
        ts_df = ts_df[ts_df['Date'] >= date_start]
    if date_end is not None:
        date_end = pd.to_datetime(date_end)
        ts_df = ts_df[ts_df['Date'] <= date_end]
    


    ts_df = ts_df.sort_values(by='Date', ascending=True).reset_index().drop(columns='index')
    ts_df['Adj Close'] = ts_df['Adj Close'].astype('float').round(4)
    ts_df['Ticker'] = ticker

    time.sleep(2)
    
    return ts_df

In [105]:
def calculate_returns(df, frequency=1):
    """ Calculate the log returns of a security given a df with its prices over a time period

    Args:
        df (dataframe): A dataframe with columns Date, Ticker, Volume, and Adj Close
        frequency (int, optional): How often you want to calculate a return Defaults to 1.

    Returns:
        df: The dataframe from the start with an additional 'Log Return' column which is the log return over the interval for each row
    """
    returns_df = df[['Date', 'Ticker', 'Volume', 'Adj Close']].copy()
    returns_df['Adj Close'] = returns_df['Adj Close'].astype('float')
    returns_df['Log Return'] = np.log(
        returns_df['Adj Close'] / returns_df['Adj Close'].shift(frequency)
    )
    returns_df = returns_df.dropna()

    # If frequency > 1, keep only every `frequency`-th row
    if frequency > 1:
        returns_df = returns_df.iloc[frequency-1::frequency].reset_index(drop=True)

    return returns_df



In [106]:
def calculate_sigma_matrix(df_list):
    return_array = []

    for df in df_list:
        if df is not None:
            print(df.shape)
            df = df.iloc[:60]
            return_array.append(df['Log Return'])

    r_arr = np.column_stack(return_array)
    Sigma = np.cov(r_arr, rowvar=False, ddof=1)

    return Sigma


In [143]:
def calculate_expected_returns(rf, Sigma, spy_returns_df=None, market_expected_return=None):
    """Monthly CAPM expected returns.

    E[R] = rf + beta * (E[R_m] - rf),  with  beta = Cov(R_i, R_m) / Var(R_m).

    SPY is asset 0 in Sigma:
    beta_i = Sigma[i, 0] / Sigma[0, 0].

    E[R_m] is either passed explicitly or estimated as the mean of the last 60 SPY
    log returns (same window as ``calculate_sigma_matrix``).

    Parameters
    ----------
    rf : float
        Monthly risk-free rate (same units as log returns).
    Sigma : ndarray, shape (n, n)
        Covariance matrix of log returns; column/row 0 is SPY.
    spy_returns_df : DataFrame, optional
        Must include ``'Log Return'``. Used if ``market_expected_return`` is None.
    market_expected_return : float, optional
        Direct monthly expected return for SPY; overrides ``spy_returns_df``.

    Returns
    -------
    ndarray, shape (n,)
        CAPM expected monthly log returns (aligned with columns of Sigma).
    """
    Sigma = np.asarray(Sigma, dtype=float)
    var_m = Sigma[0, 0]
    if var_m <= 0:
        raise ValueError("Sigma[0, 0] (market variance) must be positive.")
    betas = Sigma[:, 0] / var_m
    print("betas", betas)

    if market_expected_return is not None:
        E_rm = market_expected_return
    else:
        if spy_returns_df is None:
            raise ValueError("Pass spy_returns_df or market_expected_return.")
        E_rm = spy_returns_df['Log Return'].iloc[60]
        print("HI")
    

    mu = rf + betas * (E_rm - rf)
    mu = mu.reshape(-1, 1)
    print("mu", mu)
    return mu



In [108]:
def calculate_tangent_port(Sigma, mu, rf, cash_pct):
    """Max-Sharpe on risky assets (excludes Σ row/col 0 = SPY). Appends cash weight 0 last for portfolio code."""
    Sigma = np.asarray(Sigma, dtype=float)
    mu = np.asarray(mu, dtype=float).reshape(-1, 1)
    Sigma_r = Sigma[1:, 1:]
    mu_r = mu[1:, :]
    n = mu_r.shape[0]

    Sigma_inv = np.linalg.inv(Sigma_r)
    excess = mu_r - rf * np.ones((n, 1))

    top = Sigma_inv @ excess
    bot = np.ones((1, n)) @ Sigma_inv @ excess

    tang_wt = top / bot
    tang_w = tang_wt.T

    ER = tang_w @ mu_r
    Var = tang_w @ Sigma_r @ tang_wt
    Vol = np.sqrt(Var)
    SR = (ER - rf) / Vol

    tang_w = tang_w.reshape(-1, 1)
    # Cash leg (zero weight to start); last row matches fake cash price = 1 in p0/p1/p2
    tang_w = np.vstack([tang_w * (1-cash_pct), [[cash_pct]]])

    print(f"Weights (risky + cash) {tang_w.round(4)}")

    print("")
    print(f"ER: {ER}")
    print(f"Var: {Var}")
    print(f"Vol: {Vol}")
    print(f"SR: {SR}")

    return tang_w

In [132]:
def calculate_p0_p1_p2(df_list, rf):
    """Prices at months 62, 63, 64; skips SPY (index 0). Appends cash price 1 so lengths match tangent_w (cash weight 0)."""
    p0 = []
    p1 = []
    p2 = []
    for df in df_list[1:]:
        p0.append(df['Adj Close'].iloc[61]) # 62nd months price
        p1.append(df['Adj Close'].iloc[62]) # 63rd months price
        p2.append(df['Adj Close'].iloc[63]) # 64th months price

    p0 = np.array(p0).reshape(-1, 1)
    p1 = np.array(p1).reshape(-1, 1)
    p2 = np.array(p2).reshape(-1, 1)
    p0 = np.vstack([p0, [[1.0]]])
    p1 = np.vstack([p1, [[1.0 * (1+rf)]]])
    p2 = np.vstack([p2, [[1.0 * (1+rf)**2]]])
    return p0, p1, p2

In [110]:
def initial_values(w0, p0, k0, I):
    n = len(w0.ravel())             # -----   Setting up the LinAlg System of Equations ----- 
    sign = np.sign(w0).ravel()
    D = np.diag(sign / k0.ravel())
    ones = np.ones((1, n))
    zero = np.array([[0]])

    A = np.block([[D, -1*w0], [ones, zero]])
    C = np.zeros((n + 1, 1))
    C[-1, 0] = I

    m = np.linalg.inv(A) @ C        # -----   The answer to our system of equations which contains the weights and an extra value at the end
    m0 = m[0:n].round(3)            # Removing the extra value at the end and rounding to 3 decimals
    asset_sum = m[n+1:,].round(3)

    if np.sum(m0) > I + (I/100) or np.sum(m0) < I - (I/100):    # Just a quick check to see that our margin ~=~ I
        print("Margin Allocations != I")

    e0 = m0
    a0 = (e0/k0) * np.sign(w0)          # Assets at T=0
    l0 = np.where(w0>0, a0-e0, a0*-1)   # Liabilitis at T=0
    s0 = a0 / p0                        # Shares at T=0
    er0 = k0                            # Equity Ratio at t=0
    unlocked_cash = a0[-1][-1]  
    c0 = np.array([unlocked_cash, 0])   # Locked cash is the second element, which is 0 initially

    f0 = np.zeros((n, 1)) # Initializing fees, starts out at 0 obviously

    if True:
        print("----------  Initial Values  ----------")
        print("# Shares", s0.ravel().round(3))
        print("$ Assets", a0.ravel().round(3))
        print("$ Equity", e0.ravel().round(3)) 
        print("$ Liability", l0.ravel().round(3))
        print("M Ratio", mr0.ravel().round(3))
        print("$ Margin", m0.ravel().round(3))
        print(f"Unlocked C {c0[0]} - Locked C {c0[1]}")

    return s0, a0, e0, l0, er0, m0, c0, f0

In [164]:
def price_change(p0, p1, s, a, e, l, er, m, c, f, rf):
    a1 = p1*s
    l1 = np.where(a1>0, l, l + (p0-p1)*s)       # For longs L stays same, for shorts L changes by the change in short val(a) (same direction)
    e1 = np.where(a1>0, e + (a1 - a), e - (p0-p1)*s)   # For Longs e is a-l, for shorts e changes same as L but opposite direction
    er1 = e1/a1 * np.sign(a1)

    
    todays_long_fee = np.where(a>0, l, 0) * long_fee    # The sum of our long liabilities * fee
    todays_short_fee = np.where(a<0, l, 0) * short_fee  # The sum of our short liabilities * fee
    # We need to subtract these fees from equity, gonna wait until we get further clarification
    todays_total_fee = todays_long_fee + todays_short_fee
    e1 = e1 - todays_total_fee
    
    f1 = f + todays_total_fee  # Previous value plus todays fee # ^^

    # Cash Specific 
    e1[-1] = e[-1] * (1 + rf) # Equity for cash should not be changing after a price shift, even if mratio != 1

    if True:
        print("----------  After Price Change  ----------")
        print("# Shares", s.ravel().round(3))
        print("$ Assets", a1.ravel().round(3))
        print("$ Equity", e1.ravel().round(3)) 
        print("$ Liability", l1.ravel().round(3))
        print("M Ratio", er1.ravel().round(3))
        print("$ Margin", m.ravel().round(3))
        print("$ Fees", f1.ravel().round(3))
        print(f"Unlocked C {c[0]} - Locked C {c[1]}")

    return s, a1, e1, l1, er1, m, c, f1
    

In [112]:
def margin_call(s, a, e, l, er, m, c, f):
    ulcash = c[0]   # Pulling our unlocked and locked cash values from the c vector
    lcash = c[1]
    
    mc = np.where(er<0.25, "YES", "NO")
    en = 0.25 * a * np.sign(w0)     # "Equity Needed" or the Minimum Equity Level: What is the least amount of equity we need to be over/at 0.25
    eadd = np.where(er<0.25, en - e, 0)     # "Equity to Add": How much equity we need to add to be at the Minimum level above. 

    print("Equity Right Now")
    print(e.ravel().round(3))
    print("Total Equity Required")
    print(en.ravel().round(3))
    print("Equity Need to Add")
    print(eadd.ravel().round(3))

    if np.sum(eadd) > 0:    # If no positions need equity eadd is all 0, so this doesn't trigger
        print("Need Margin Call")
        lcash += np.sum(eadd)  # The sum of the equity we need is added to our locked cash
        e = e + eadd        # We add the equity to add to our equity to level off the equity levels to atleast 0.25 levels
        ulcash -= np.sum(eadd)  # Inverse to locked cash, we lose x amount of unlocked cash to whatever goes to eadd
        m = m + eadd            # We add the equity we are adding to our margin
        m[-1] = m[-1] - np.sum(eadd)  # Cash margin is old cash margin - the cash we need for margin (old margin - change locked cash)
        e[-1] = e[-1] - np.sum(eadd)  # Same idea as margin

        if ulcash < 0: # If we need more cash then we have in unlocked cash
            cadd = -1 * ulcash      # This is the amount of new cash we need
            print(f"We need to add ${cadd.round(2)} of cash")
            e[-1] = 0   # Our cash equity is 0
            ulcash = 0
            m[-1] = 0   # Margin equity also 0
            a[-1] = a[-1] + cadd    # Add the new cash to assets
            s[-1] = s[-1] + cadd    # Add the new cash to shares
        else:
            print("Did not need to add more cash")

        c = np.array([float(np.asarray(ulcash).squeeze()), float(np.asarray(lcash).squeeze())], dtype=float) # Some BS to get cash to fit in our 2, array
        
        
    else:
        print("No margin call needed")

    er = e / a * np.sign(a) # Regular equity ratio calculation

    if True:
        print("")
        print("----------  After Margin Call  ----------")
        print("# Shares", s.ravel().round(3))
        print("$ Assets", a.ravel().round(3))
        print("$ Equity", e.ravel().round(3)) 
        print("$ Liability", l.ravel().round(3))
        print("M Ratio", er.ravel().round(3))
        print("$ Margin", m.ravel().round(3))
        print(f"Unlocked C {c[0].round(2)} - Locked C {c[1].round(2)}")

    return s, a, e, l, er, m, c, f

In [165]:
def rebalance2(p, s, a, e, l, er, m, c, f):
    w1 = a / a.sum()
    dist = np.linalg.norm(w1 - w0) / np.linalg.norm(w0)

    if dist > 0.05:
        print(f"Distance is {dist} YES rebalance (rebalance2).")

        ulcash = float(np.asarray(c[0]).squeeze())
        lcash = float(np.asarray(c[1]).squeeze())

        V = np.sum(a)
        a_star = w0 * V
        d = a_star - a

        dc = float(np.asarray(d[-1]).squeeze())
        if dc < 0:
            lam = min(1.0, ulcash / (-dc))
        else:
            lam = 1.0
        lam = max(0.0, lam)

        if lam == 0:
            # Full target would drain unlocked cash; rebalance risky sleeve only (cash unchanged).
            # Non-cash weights renormalized to sum to 1: w_risky = w0[:-1] / sum(w0[:-1])
            sum_w_risky = float(np.sum(w0[:-1]))
            if abs(sum_w_risky) < 1e-12:
                a1 = np.array(a, copy=True)
            else:
                w_risky = w0[:-1] / sum_w_risky
                V_risky = np.sum(a[:-1])
                a1 = np.vstack((w_risky * V_risky, a[-1:]))
            print("rebalance2 λ=0 → risky-only (w_risky = w0[:-1]/sum(w0[:-1]), targets = w_risky * sum(a[:-1]))")
        else:
            a1 = a + lam * d

        l1 = np.where(a1>0, l + ((a1 - a) * (1 - er)), np.abs(a1))
        e1 = np.where(a1>0, e + ((a1 - a) * (er)), a1 * er)
        e1 = np.abs(e1)
        s1 = a1 / p
        delta_s = s1 - s
        rebal_val = delta_s * p
        #l1 = np.where(w1 > 0, a1 - e1, np.abs(a1))
        m1 = np.abs(np.where(w1 > 0, m + rebal_val * er, m - rebal_val * er))
        er1 = e1 / a1 * np.sign(a1)

        delta_cash = a1[-1] - a[-1]
        ulcash += float(np.asarray(delta_cash).squeeze())
        m1[-1] = m[-1] + delta_cash
        e1[-1] = e[-1] + delta_cash
        l1[-1] = 0

        if lcash < 0:
            lcash = 0

        try:
            c1 = np.array([ulcash, lcash])
        except Exception:
            c1 = np.array([ulcash, [lcash]])

        print(f"rebalance2 λ (cash-feasible) = {lam}")
    else:
        a1 = a
        e1 = e
        s1 = s
        l1 = l
        m1 = m
        c1 = c
        er1 = er
        print(f"Distance is {dist} NO rebelance.")

    if True:
        print("")
        print("----------  After Rebalance  ----------")
        print("# Shares", s1.ravel().round(3))
        print("$ Assets", a1.ravel().round(3))
        print("$ Equity", e1.ravel().round(3))
        print("$ Liability", l1.ravel().round(3))
        print("M Ratio", er.ravel().round(3))
        print("$ Margin", m1.ravel().round(3))
        print(f"Unlocked C {c1[0].round(2)} - Locked C {c1[1].round(2)}")

    return s1, a1, e1, l1, er1, m1, c1, f

In [166]:
def calculate_return_drawdown(p0e, p11e, p12e, p21e, p22e, p1_eadd, p2_eadd):
    
    r1 = ln((np.sum(p11e) - np.sum(p1_eadd)) / np.sum(p0e))
    r2 = ln((np.sum(p21e) - np.sum(p2_eadd)) / np.sum(p12e))

    DD1 = 1 - (np.sum(p11e) - np.sum(p1_eadd)) / np.sum(p0e)
    if np.sum(p0e) > np.sum(p11e) - np.sum(p1_eadd):
        DD2 = 1 - (np.sum(p21e) - np.sum(p2_eadd)) / np.sum(p0e)
    else:
        DD2 = 1 - (np.sum(p21e) - np.sum(p2_eadd)) / (np.sum(p12e) - np.sum(p1_eadd))
        
    max_drawdown = max(0, DD1, DD2)
    
    return r1, r2, max_drawdown

In [147]:
""" -+-+-+-+-+  INPUTS  -+-+-+-+-+ """


# ALWAYS SPY OR INDEX FIRST
tickers = ['SPY', 'JPM', 'AAPL', 'MSFT', 'GS']

prices_df_dict = {}
for ticker in tickers:
    prices_df_dict[ticker] = get_prices(ticker, size='full', date_start='2000-01-01', date_end='2026-03-31')

rf = ln(1 + 0.05/12)

I = 10000

# One margin ratio per position: risky (tickers[1:]), then cash (1.0)
mr0 = np.array([[0.5] for _ in tickers[1:]] + [[1.0]])

long_fee = 0.0001
short_fee = 0.0001

In [162]:
""" -+-+-+-+-+  RANDOM START DATES GENERATOR  -+-+-+-+-+ """

date_start_list = []
for i in range(10):
    year = random.randint(2000, 2020)
    month = random.randint(1, 12)
    d = pd.Timestamp(year=year, month=month, day=1)
    date_start_list.append(d)

if True: # Change to True and add dates to the list to test specific dates
    date_start_list = [] 
    date_start_strings = ['2010-01-01', '2016-01-01', '2020-01-01']
    for date_start in date_start_strings:
        date_start_list.append(pd.to_datetime(date_start))
        
print(date_start_list)


[Timestamp('2010-01-01 00:00:00'), Timestamp('2016-01-01 00:00:00'), Timestamp('2020-01-01 00:00:00')]


In [ ]:
""" -+-+-+-+-+  MAIN  -+-+-+-+-+ """

period_returns = []
test_returns = []
drawdowns = []
spy_returns = []

returns_df_list = []


for date_start in date_start_list:
    print("Date Start", date_start)
    data_df_list = []
    end_date = date_start + pd.DateOffset(months=64)
    
    for price_df in prices_df_dict.values():
        filtered_df = price_df[price_df['Date'] >= date_start]
        filtered_df = filtered_df[filtered_df['Date'] <= end_date]
        data_df_list.append(filtered_df)
    
    returns_df_list = [calculate_returns(data_df) for data_df in data_df_list]
    sigma_matrix = calculate_sigma_matrix(returns_df_list)
    print(sigma_matrix)
    expected_returns_arr = calculate_expected_returns(rf, sigma_matrix, returns_df_list[0])
    w0 = calculate_tangent_port(sigma_matrix, expected_returns_arr, rf, 0.05)
    p0, p1, p2 = calculate_p0_p1_p2(data_df_list, rf)
    
    s, a, e, l, er, m, c, f = initial_values(w0, p0, mr0, I)
    p0e = e
    p0_lc = c[1]
    s, a, e, l, er, m, c, f = price_change(p0, p1, s, a, e, l, er, m, c, f, rf)
    s, a, e, l, er, m, c, f = margin_call(s, a, e, l, er, m, c, f)
    p1_eadd = c[1] - p0_lc
    p11e = e
    s, a, e, l, er, m, c, f = rebalance2(p1, s, a, e, l, er, m, c, f)
    p12e = e
    p1_lc = c[1]
    s, a, e, l, er, m, c, f = price_change(p1, p2, s, a, e, l, er, m, c, f, rf)
    s, a, e, l, er, m, c, f = margin_call(s, a, e, l, er, m, c, f)
    p2_eadd = c[1] - p1_lc
    p21e = e
    s, a, e, l, er, m, c, f = rebalance2(p2, s, a, e, l, er, m, c, f)
    p22e = e

    r1, r2, max_drawdown = calculate_return_drawdown(p0e, p11e, p12e, p21e, p22e, p1_eadd, p2_eadd)
    period_returns.append([r1, r2])
    test_returns.append(r1 + r2)
    drawdowns.append(max_drawdown)

    

    spy_loop_returns = returns_df_list[0]['Log Return']
    spy_2m_return = spy_loop_returns.iloc[-1] + spy_loop_returns.iloc[-2]
    spy_returns.append(spy_2m_return)



Date Start 2010-01-01 00:00:00
(63, 5)
(63, 5)
(63, 5)
(63, 5)
(63, 5)
[[0.00137907 0.00228836 0.00114073 0.00142671 0.00220851]
 [0.00228836 0.00646331 0.00135606 0.00281683 0.00562395]
 [0.00114073 0.00135606 0.00505865 0.00140723 0.00130127]
 [0.00142671 0.00281683 0.00140723 0.00379794 0.00253888]
 [0.00220851 0.00562395 0.00130127 0.00253888 0.00695152]]
betas [1.         1.65934906 0.82717563 1.03454841 1.60145059]
HI
mu [[0.05468219]
 [0.08799526]
 [0.04595038]
 [0.05642772]
 [0.08506998]]
Weights (risky + cash) [[0.3473]
 [0.2053]
 [0.2294]
 [0.168 ]
 [0.05  ]]

ER: [[0.07076914]]
Var: [[0.00344084]]
Vol: [[0.0586587]]
SR: [[1.13557105]]
----------  Initial Values  ----------
# Shares [145.847 136.892 116.356  20.999 952.381]
$ Assets [6615.662 3910.148 4369.988 3199.438  952.381]
$ Equity [3307.831 1955.074 2184.994 1599.719  952.381]
$ Liability [3307.831 1955.074 2184.994 1599.719    0.   ]
M Ratio [0.5 0.5 0.5 0.5 1. ]
$ Margin [3307.831 1955.074 2184.994 1599.719  952.381]

In [160]:
""" -+-+-+-+-+  MAIN 2  -+-+-+-+-+ """

i = 0
for period_return in period_returns:
    i += 1
    period_sharpe = (np.mean(period_return) - rf) / np.std(period_return)
    print(f"Start Date {date_start_list[i-1].strftime('%Y-%m-%d')} r1: {period_return[0]:.3f} r2: {period_return[1]:.3f} SR: {period_sharpe:.3f} max_dd: {drawdowns[i-1]:.3f}")


Start Date 2019-11-01 r1: 0.056 r2: -0.022 SR: 0.340 max_dd: 0.021
Start Date 2020-10-01 r1: 0.006 r2: -0.113 SR: -0.972 max_dd: 0.099
Start Date 2002-09-01 r1: -0.117 r2: -0.002 SR: -1.109 max_dd: 0.112
Start Date 2001-11-01 r1: 0.085 r2: -0.086 SR: -0.057 max_dd: 0.082
Start Date 2007-08-01 r1: -0.030 r2: -0.057 SR: -3.608 max_dd: 0.089
Start Date 2015-08-01 r1: -0.030 r2: 0.225 SR: 0.730 max_dd: 0.030
Start Date 2013-05-01 r1: 0.127 r2: 0.118 SR: 24.740 max_dd: 0.000
Start Date 2011-07-01 r1: -0.005 r2: 0.094 SR: 0.808 max_dd: 0.005
Start Date 2019-03-01 r1: 0.141 r2: 0.075 SR: 3.142 max_dd: 0.000
Start Date 2007-01-01 r1: 0.165 r2: -0.068 SR: 0.383 max_dd: 0.065


In [161]:
""" -+-+-+-+-+  MAIN 3  -+-+-+-+-+ """

average_2m_port_return = np.mean(test_returns).round(5)
average_2m_spy_return = (np.mean(spy_returns)).round(5)

# Compute t test of port return vs spy return
t_stat, p_val = stats.ttest_rel(test_returns, spy_returns)

print(f"Avg. 2m Port Ret. {average_2m_port_return} vs Avg. 2m SPY Ret. {average_2m_spy_return} with difference {(average_2m_port_return - average_2m_spy_return):.4f}")
print(f"Statistical Significance of Port vs SPY: p_val = {p_val:.4f}, t_stat = {t_stat:.4f}")


Avg. 2m Port Ret. 0.0561 vs Avg. 2m SPY Ret. 0.0198 with difference 0.0363
Statistical Significance of Port vs SPY: p_val = 0.2836, t_stat = 1.1403


In [ ]:
""" Below Goes Through Each function/step in the process 1 by 1 """
""" ANYTHING BELOW HERE IS NOT USED IN THE MAIN PROCESS """
""" IF I AM ASKING TO REVIEW MY CODE. DO NOT PAY ATTENTION TO ANYTHING BELOW HERE """

' IF I AM ASKING TO REVIEW MT CODE. DO NOT PAY ATTENTION TO ANYTHING BELOW HERE '

In [167]:
date_start = pd.to_datetime('2000-01-01')
date_end = date_start + pd.DateOffset(months=64)
data_df_list = []
for ticker in tickers:
    ticker_data_df = get_prices(ticker, size='full', date_start=date_start, date_end=date_end)
    #time.sleep(2)
    data_df_list.append(ticker_data_df)
    print(ticker_data_df.head(2))



        Date      Open      High       Low     Close  Adj Close     Volume  \
0 2000-01-31  148.2500  148.2500  135.0000  139.5625    87.6045  156770800   
1 2000-02-29  139.7500  144.5625  132.7187  137.4375    86.2707  186938300   

  Dividend Ticker  
0   0.0000    SPY  
1   0.0000    SPY  
        Date     Open     High      Low    Close  Adj Close     Volume  \
0 2000-01-31  74.7500  81.5000  68.2500  80.6900    24.5490  109109004   
1 2000-02-29  82.4400  86.3700  74.5600  79.6200    24.2234   88645202   

  Dividend Ticker  
0   0.4100    JPM  
1   0.0000    JPM  
        Date      Open      High      Low     Close  Adj Close     Volume  \
0 2000-01-31  104.8700  121.5000  86.5000  103.7500     0.7771  112099800   
1 2000-02-29  104.0000  119.9400  97.0000  114.6200     0.8586   65355200   

  Dividend Ticker  
0   0.0000   AAPL  
1   0.0000   AAPL  
        Date      Open      High      Low    Close  Adj Close     Volume  \
0 2000-01-31  117.3700  118.6200  94.8700  97.8700    

In [168]:
returns_df_list = []

for data_df in data_df_list:
    return_df = calculate_returns(data_df)
    returns_df_list.append(return_df)
    print(return_df.head(2))


        Date Ticker     Volume  Adj Close  Log Return
1 2000-02-29    SPY  186938300    86.2707   -0.015342
2 2000-03-31    SPY  247594900    94.6299    0.092483
        Date Ticker     Volume  Adj Close  Log Return
1 2000-02-29    JPM   88645202    24.2234   -0.013352
2 2000-03-31    JPM  110318738    26.5265    0.090825
        Date Ticker    Volume  Adj Close  Log Return
1 2000-02-29   AAPL  65355200     0.8586    0.099734
2 2000-03-31   AAPL  77663900     1.0173    0.169604
        Date Ticker      Volume  Adj Close  Log Return
1 2000-02-29   MSFT   667243800    27.2912   -0.090853
2 2000-03-31   MSFT  1014093800    32.4459    0.173010
        Date Ticker    Volume  Adj Close  Log Return
1 2000-02-29     GS  18721100    64.1175    0.009451
2 2000-03-31     GS  35059300    72.9900    0.129605


In [169]:
sigma_matrix = calculate_sigma_matrix(returns_df_list)
print(sigma_matrix)

(63, 5)
(63, 5)
(63, 5)
(63, 5)
(63, 5)
[[0.00219828 0.00400701 0.00435462 0.00334528 0.00351198]
 [0.00400701 0.01244134 0.00770393 0.00569544 0.00740501]
 [0.00435462 0.00770393 0.03549711 0.00989029 0.01017724]
 [0.00334528 0.00569544 0.00989029 0.01470771 0.00505687]
 [0.00351198 0.00740501 0.01017724 0.00505687 0.01109038]]


In [170]:
expected_returns_arr = calculate_expected_returns(rf, sigma_matrix, returns_df_list[0])
print(expected_returns_arr)

betas [1.         1.82279754 1.98092642 1.521774   1.59760511]
HI
mu [[0.02068865]
 [0.03429001]
 [0.03690398]
 [0.0293139 ]
 [0.03056744]]
[[0.02068865]
 [0.03429001]
 [0.03690398]
 [0.0293139 ]
 [0.03056744]]


In [172]:
w0 = calculate_tangent_port(sigma_matrix, expected_returns_arr, rf, 0.05)

Weights (risky + cash) [[0.4141]
 [0.0372]
 [0.2144]
 [0.2843]
 [0.05  ]]

ER: [[0.0321556]]
Var: [[0.00857276]]
Vol: [[0.09258921]]
SR: [[0.30238505]]


In [174]:
p0, p1, p2 = calculate_p0_p1_p2(data_df_list, rf)
p0, p1, p2

(array([[20.8851],
        [ 1.3441],
        [17.3484],
        [78.2967],
        [ 1.    ]]),
 array([[19.7709    ],
        [ 1.2485    ],
        [16.6658    ],
        [79.153     ],
        [ 1.00415801]]),
 array([[20.4836    ],
        [ 1.0804    ],
        [17.445     ],
        [77.0346    ],
        [ 1.00833331]]))

In [175]:
s, a, e, l, er, m, c, f = initial_values(w0, p0, mr0, I)

----------  Initial Values  ----------
# Shares [377.682 527.644 235.393  69.153 952.381]
$ Assets [7887.92   709.206 4083.686 5414.426  952.381]
$ Equity [3943.96   354.603 2041.843 2707.213  952.381]
$ Liability [3943.96   354.603 2041.843 2707.213    0.   ]
M Ratio [0.5 0.5 0.5 0.5 1. ]
$ Margin [3943.96   354.603 2041.843 2707.213  952.381]
Unlocked C 952.381 - Locked C 0.0


In [176]:
s, a, e, l, er, m, c, f = price_change(p0, p1, s, a, e, l, er, m, c, f, rf)

----------  After Price Change  ----------
# Shares [377.682 527.644 235.393  69.153 952.381]
$ Assets [7467.107  658.763 3923.007 5473.641  956.341]
$ Equity [3522.753  304.125 1880.96  2766.158  956.341]
$ Liability [3943.96   354.603 2041.843 2707.213    0.   ]
M Ratio [0.472 0.462 0.48  0.505 1.   ]
$ Margin [3943.96   354.603 2041.843 2707.213  952.381]
$ Fees [0.394 0.035 0.204 0.271 0.   ]
Unlocked C 952.381 - Locked C 0.0


In [177]:
s, a, e, l, er, m, c, f = margin_call(s, a, e, l, er, m, c, f)

Equity Right Now
[3522.753  304.125 1880.96  2766.158  956.341]
Total Equity Required
[1866.777  164.691  980.752 1368.41   239.085]
Equity Need to Add
[0. 0. 0. 0. 0.]
No margin call needed

----------  After Margin Call  ----------
# Shares [377.682 527.644 235.393  69.153 952.381]
$ Assets [7467.107  658.763 3923.007 5473.641  956.341]
$ Equity [3522.753  304.125 1880.96  2766.158  956.341]
$ Liability [3943.96   354.603 2041.843 2707.213    0.   ]
M Ratio [0.472 0.462 0.479 0.505 1.   ]
$ Margin [3943.96   354.603 2041.843 2707.213  952.381]
Unlocked C 952.38 - Locked C 0.0


In [178]:
s, a, e, l, er, m, c, f = rebalance2(p1, s, a, e, l, er, m, c, f)

Distance is 0.02895986975095804 NO rebelance.

----------  After Rebalance  ----------
# Shares [377.682 527.644 235.393  69.153 952.381]
$ Assets [7467.107  658.763 3923.007 5473.641  956.341]
$ Equity [3522.753  304.125 1880.96  2766.158  956.341]
$ Liability [3943.96   354.603 2041.843 2707.213    0.   ]
M Ratio [0.472 0.462 0.479 0.505 1.   ]
$ Margin [3943.96   354.603 2041.843 2707.213  952.381]
Unlocked C 952.38 - Locked C 0.0


In [179]:
s, a, e, l, er, m, c, f = price_change(p1, p2, s, a, e, l, er, m, c, f, rf)

----------  After Price Change  ----------
# Shares [377.682 527.644 235.393  69.153 952.381]
$ Assets [7736.281  570.066 4106.425 5327.148  960.317]
$ Equity [3791.532  215.392 2064.174 2619.394  960.317]
$ Liability [3943.96   354.603 2041.843 2707.213    0.   ]
M Ratio [0.49  0.378 0.503 0.492 1.   ]
$ Margin [3943.96   354.603 2041.843 2707.213  952.381]
$ Fees [0.789 0.071 0.408 0.541 0.   ]
Unlocked C 952.381 - Locked C 0.0


In [180]:
s, a, e, l, er, m, c, f = margin_call(s, a, e, l, er, m, c, f)

Equity Right Now
[3791.532  215.392 2064.174 2619.394  960.317]
Total Equity Required
[1934.07   142.517 1026.606 1331.787  240.079]
Equity Need to Add
[0. 0. 0. 0. 0.]
No margin call needed

----------  After Margin Call  ----------
# Shares [377.682 527.644 235.393  69.153 952.381]
$ Assets [7736.281  570.066 4106.425 5327.148  960.317]
$ Equity [3791.532  215.392 2064.174 2619.394  960.317]
$ Liability [3943.96   354.603 2041.843 2707.213    0.   ]
M Ratio [0.49  0.378 0.503 0.492 1.   ]
$ Margin [3943.96   354.603 2041.843 2707.213  952.381]
Unlocked C 952.38 - Locked C 0.0


In [181]:
s, a, e, l, er, m, c, f = rebalance2(p2, s, a, e, l, er, m, c, f)

Distance is 0.01575017187675905 NO rebelance.

----------  After Rebalance  ----------
# Shares [377.682 527.644 235.393  69.153 952.381]
$ Assets [7736.281  570.066 4106.425 5327.148  960.317]
$ Equity [3791.532  215.392 2064.174 2619.394  960.317]
$ Liability [3943.96   354.603 2041.843 2707.213    0.   ]
M Ratio [0.49  0.378 0.503 0.492 1.   ]
$ Margin [3943.96   354.603 2041.843 2707.213  952.381]
Unlocked C 952.38 - Locked C 0.0
